<!-- notebook-header -->
# CNNs: Fundamentos e Arquiteturas

**Modulo:** 05 - Dominios Aplicados / 05A - Computer Vision  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Convolucao, pooling, feature maps, receptive field e blocos de CNNs.


# CNNs: Fundamentos e Arquiteturas

**Modulo 5A -- Visao Computacional | Notebook 1 de 5**

Neste notebook, construiremos a intuicao por tras das Redes Neurais Convolucionais (CNNs),
desde a operacao de convolucao ate arquiteturas modernas como ResNet e EfficientNet.
Entenderemos *por que* CNNs sao tao eficazes para imagens e como cada componente contribui.

## Pre-requisitos e Fio Narrativo

**Antes deste notebook voce deve ter estudado:**
- Fundamentos de redes neurais (4_1) -- neuronios, camadas, backpropagation
- Arquiteturas deep learning (4_2) -- conceito de camadas convolucionais
- Treinamento (4_3) -- BatchNorm, Dropout, learning rate scheduling

**Fio narrativo:** Ate agora, tratamos dados como vetores "planos". Mas imagens tem
*estrutura espacial* -- pixels vizinhos sao correlacionados. CNNs exploram essa estrutura
com **compartilhamento de pesos** e **localidade**, reduzindo parametros drasticamente
e aprendendo hierarquias de features automaticamente.

**O que vamos construir:** Partiremos de uma operacao simples (filtro deslizante) ate
arquiteturas que ganham competicoes internacionais, entendendo cada decisao de design.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

# PyTorch (disponivel no seu ambiente Jupyter)
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.models as models
    HAS_TORCH = True
    print(f'PyTorch version: {torch.__version__}')
    print(f'GPU disponivel: {torch.cuda.is_available()}')
except ImportError:
    HAS_TORCH = False
    print('PyTorch nao disponivel -- celulas PyTorch serao puladas')

np.random.seed(42)
print('Imports carregados com sucesso!')

In [ ]:
# Detecao automatica de hardware (CPU / CUDA / MPS Apple Silicon)
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Apple Silicon MPS detectado')
else:
    device = torch.device('cpu')
    print('Executando em CPU (treinamento sera mais lento, mas funciona)')
print(f'Device selecionado: {device}')

## 1. Por que CNNs para Imagens?

### Analogia: O Inspetor de Qualidade

Imagine um inspetor de qualidade numa fabrica de tecidos. Ele nao olha o tecido inteiro
de uma vez -- ele usa uma *lupa* que desliza sobre o tecido, procurando defeitos locais.
Se encontrar um defeito num canto, sabe reconhece-lo em qualquer outro canto tambem.

Uma CNN funciona exatamente assim:
1. **A lupa** = filtro/kernel (3x3 ou 5x5 pixels)
2. **Deslizar** = convolucao (varrer toda a imagem)
3. **Mesmo defeito em qualquer posicao** = compartilhamento de pesos (invariancia translacional)
4. **Lupa mais potente** = camadas mais profundas (features mais abstratas)

### Definicao Formal

Uma CNN e uma rede neural que usa **operacoes de convolucao** em pelo menos uma camada,
explorando tres propriedades fundamentais das imagens:
- **Localidade:** pixels vizinhos sao mais correlacionados que distantes
- **Estacionariedade:** padroes podem aparecer em qualquer posicao
- **Composicionalidade:** features complexas sao combinacoes de features simples

### Por que em ML: A Revolucao da Visao Computacional

Antes de CNNs (2012), visao computacional usava features manuais (SIFT, HOG). AlexNet
mostrou que features *aprendidas* por CNNs superam features manuais em todas as tarefas.
Hoje, CNNs sao a base de: reconhecimento facial, carros autonomos, diagnostico medico
por imagem, controle de qualidade industrial, e muito mais.

In [ ]:

# Demonstração de parâmetros: MLP vs CNN
image_size = 28  # MNIST
channels = 1
hidden_units = 128

# MLP parameters
mlp_input = image_size * image_size * channels  # 784
mlp_hidden_to_hidden = hidden_units * hidden_units
mlp_params = mlp_input * hidden_units + mlp_hidden_to_hidden + hidden_units

# CNN parameters (simples com 1 conv layer)
kernel_size = 3
out_channels = 32
cnn_params = (kernel_size * kernel_size * channels * out_channels) + out_channels

print('Comparação de Parâmetros: MNIST 28x28')
print(f'MLP (entrada -> {hidden_units} hidden -> {hidden_units} hidden):')
print(f'  - Input layer: {mlp_input * hidden_units:,} parâmetros')
print(f'  - Hidden layers: {mlp_hidden_to_hidden + hidden_units:,} parâmetros')
print(f'  - Total: {mlp_params:,} parâmetros')
print()
print(f'CNN (1 Conv2d layer 3x3, {out_channels} canais):')
print(f'  - Total: {cnn_params:,} parâmetros')
print()
print(f'Redução: {mlp_params / cnn_params:.1f}x menos parâmetros na CNN!')


### O que observar

1. A reducao de parametros e **dramatica** -- mais de 300x entre MLP e uma camada conv
2. MLP precisa de uma conexao para *cada* pixel para *cada* neuronio hidden
3. CNN compartilha o mesmo filtro em todas as posicoes da imagem
4. Para imagens maiores (224x224 RGB), a diferenca seria ainda maior: ~150.000 vs ~864 parametros

### O que concluir

O compartilhamento de pesos e a principal vantagem computacional das CNNs. Menos parametros
significam: treinamento mais rapido, menos dados necessarios, e menor risco de overfitting.

### Conexao com outros notebooks

Em 4_1 (fundamentos de redes neurais), vimos que cada neuronio fully connected
se conecta a todos os neuronios da camada anterior. Aqui, a convolucao *limita*
essas conexoes a uma vizinhanca local, o que e uma forma de **regularizacao implicita**
(conexao com 4_3 treinamento, onde vimos Dropout e outras formas de regularizacao).

## 2. Operacao de Convolucao

### Analogia: Carimbo Deslizante

Pense num carimbo de borracha com numeros (o kernel). Voce pressiona o carimbo sobre
a imagem, multiplica cada numero do carimbo pelo pixel embaixo, soma tudo, e escreve
o resultado. Depois desliza o carimbo e repete.

### Definicao Formal

A convolucao 2D entre uma entrada I e um kernel K produz:

**(I * K)[i,j] = sum_m sum_n I[i+m, j+n] * K[m,n]**

Parametros que controlam a operacao:
- **Kernel size (k):** tamanho do filtro (3x3, 5x5, 7x7)
- **Stride (s):** quantos pixels o kernel avanca por vez
- **Padding (p):** zeros adicionados nas bordas da imagem
- **Dimensao da saida:** H_out = (H_in + 2p - k) / s + 1

### Por que em ML: Detectores Aprendiveis

Cada filtro aprende a detectar um padrao especifico (bordas, texturas, cores).
Filtros de Sobel (bordas) eram *projetados manualmente*. Em CNNs, a rede *aprende*
os melhores filtros via backpropagation -- superando decadas de engenharia manual.

In [ ]:

# Simulação manual de convolução
def conv2d_manual(input_tensor, kernel, stride=1, padding=0):
    # Add padding
    if padding > 0:
        input_tensor = np.pad(input_tensor, padding, mode='constant')
    
    in_height, in_width = input_tensor.shape
    k_height, k_width = kernel.shape
    
    out_height = (in_height - k_height) // stride + 1
    out_width = (in_width - k_width) // stride + 1
    
    output = np.zeros((out_height, out_width))
    
    for i in range(out_height):
        for j in range(out_width):
            h_start = i * stride
            w_start = j * stride
            window = input_tensor[h_start:h_start+k_height, w_start:w_start+k_width]
            output[i, j] = np.sum(window * kernel)
    
    return output

# Exemplo: Detector de bordas com Sobel kernel
input_img = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=np.float32)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

output = conv2d_manual(input_img, sobel_x, stride=1, padding=0)

print('Input (4x4):')
print(input_img)
print()
print('Sobel Kernel:')
print(sobel_x)
print()
print('Output (2x2):')
print(output)
print()
print('Dimensões: (H_in=4, W_in=4, k=3, s=1, p=0)')
print(f'H_out = (4 + 0 - 3) / 1 + 1 = {output.shape[0]}')
print(f'W_out = (4 + 0 - 3) / 1 + 1 = {output.shape[1]}')


### O que observar

1. O output e menor que o input (4x4 -> 2x2) quando nao ha padding
2. O kernel de Sobel detecta **bordas verticais** (valores positivos a direita, negativos a esquerda)
3. Cada posicao do output resume a informacao de uma **vizinhanca local** do input
4. A formula H_out = (H_in + 2p - k) / s + 1 se confirma: (4 + 0 - 3) / 1 + 1 = 2

### O que concluir

A convolucao e uma **soma ponderada local** que transforma a imagem destacando padroes
especificos. Diferentes kernels detectam diferentes padroes. O tamanho do output depende
de kernel, stride e padding -- e crucial calcular essas dimensoes corretamente ao projetar
uma arquitetura.

### Conexao com outros notebooks

Em 0_3 (algebra linear -- matrizes), vimos multiplicacao elemento a elemento. A convolucao
e essencialmente isso: multiplicacao local + soma. A diferenca e que aqui o kernel *desliza*,
o que conecta com a ideia de *features compartilhadas* em toda a imagem.

In [ ]:

# Demonstrar efeito de stride e padding
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Create a simple pattern
test_input = np.random.randn(8, 8)

for i, stride in enumerate([1, 2]):
    for j, padding in enumerate([0, 1, 2]):
        ax = axes[i, j]
        
        # Calculate output size
        k = 3
        h_out = (test_input.shape[0] + 2*padding - k) // stride + 1
        w_out = (test_input.shape[1] + 2*padding - k) // stride + 1
        
        # Visualize receptive field coverage
        coverage = np.zeros_like(test_input)
        for start in range(0, test_input.shape[0] - k + 1, stride):
            for start_w in range(0, test_input.shape[1] - k + 1, stride):
                coverage[start:start+k, start_w:start_w+k] += 1
        
        im = ax.imshow(coverage, cmap='hot')
        ax.set_title(f'Stride={stride}, Padding={padding}\nOutput: {h_out}x{w_out}')
        ax.set_xlabel('Width')
        ax.set_ylabel('Height')
        plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('/tmp/stride_padding.png', dpi=100, bbox_inches='tight')
plt.show()

print('Visualização da cobertura de receptive fields com diferentes stride e padding')


### O que observar sobre Stride e Padding

1. **Stride 1, Padding 0:** output menor que input (perde bordas)
2. **Stride 1, Padding 1:** output com *mesmo tamanho* que input ("same" convolution)
3. **Stride 2:** output com **metade** do tamanho (downsampling eficiente)
4. Mais padding = mais pixels processados nas bordas = output maior
5. Mais stride = menos sobreposicao entre janelas = output menor

### O que concluir sobre Receptive Field

O **receptive field** de um neuronio e a regiao da imagem original que o influencia.
Camadas empilhadas com kernel 3x3 aumentam o receptive field: 2 camadas = 5x5, 3 camadas = 7x7.
Isso e mais eficiente que usar um unico kernel 7x7 (menos parametros, mais nao-linearidades).

In [ ]:

# Calcular receptive field em arquiteturas
def calculate_receptive_field(num_layers, kernel_size=3, stride=1, padding=1):
    rf = 1
    effective_stride = 1
    
    for layer in range(num_layers):
        rf += (kernel_size - 1) * effective_stride
        effective_stride *= stride
    
    return rf, effective_stride

print('Receptive Field em CNNs')
print('=' * 50)
print()

for num_layers in [1, 3, 5, 7]:
    rf, eff_stride = calculate_receptive_field(num_layers, kernel_size=3, stride=1)
    print(f'{num_layers} camadas (k=3, s=1, p=1):')
    print(f'  - Receptive Field: {rf}x{rf}')
    print()

print()
print('Com stride=2 a cada 2 camadas:')
for num_layers in [2, 4, 6, 8]:
    strides = [1 if i < 2 else 2 for i in range(num_layers)]
    rf = 1
    eff_stride = 1
    for s in strides:
        rf += (3 - 1) * eff_stride
        eff_stride *= s
    print(f'{num_layers} camadas (alternando s=1 e s=2):')
    print(f'  - Receptive Field: {rf}x{rf}')


## 3. Pooling: Reduzir Mantendo o Essencial

### Analogia: Resumo Executivo

Se a convolucao e como ler um livro com uma lupa, o pooling e como escrever um
**resumo executivo**: voce pega os pontos mais importantes de cada secao (max pooling)
ou a media de cada secao (average pooling), descartando detalhes irrelevantes.

### Definicao Formal

- **Max Pooling:** para cada janela kxk, retorna o valor maximo
- **Average Pooling:** retorna a media dos valores na janela
- **Global Pooling:** reduz cada feature map inteiro a um unico valor

### Por que em ML: Invariancia e Eficiencia

Max pooling fornece **invariancia translacional**: se um objeto se move poucos pixels,
o max pooling ainda captura o mesmo valor maximo. Alem disso, reduz as dimensoes
espaciais, acelerando o processamento e reduzindo overfitting.

In [ ]:

def max_pool2d(input_tensor, kernel_size=2, stride=None):
    if stride is None:
        stride = kernel_size
    
    h_in, w_in = input_tensor.shape
    k = kernel_size
    s = stride
    
    h_out = (h_in - k) // s + 1
    w_out = (w_in - k) // s + 1
    
    output = np.zeros((h_out, w_out))
    
    for i in range(h_out):
        for j in range(w_out):
            h_start = i * s
            w_start = j * s
            window = input_tensor[h_start:h_start+k, w_start:w_start+k]
            output[i, j] = np.max(window)
    
    return output

def avg_pool2d(input_tensor, kernel_size=2, stride=None):
    if stride is None:
        stride = kernel_size
    
    h_in, w_in = input_tensor.shape
    k = kernel_size
    s = stride
    
    h_out = (h_in - k) // s + 1
    w_out = (w_in - k) // s + 1
    
    output = np.zeros((h_out, w_out))
    
    for i in range(h_out):
        for j in range(w_out):
            h_start = i * s
            w_start = j * s
            window = input_tensor[h_start:h_start+k, w_start:w_start+k]
            output[i, j] = np.mean(window)
    
    return output

# Demonstração
test_input = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=np.float32)

max_out = max_pool2d(test_input, kernel_size=2, stride=2)
avg_out = avg_pool2d(test_input, kernel_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].imshow(test_input, cmap='viridis')
axes[0].set_title('Input 4x4')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))

axes[1].imshow(max_out, cmap='viridis')
axes[1].set_title('Max Pooling 2x2')
axes[1].set_xticks(range(2))
axes[1].set_yticks(range(2))

axes[2].imshow(avg_out, cmap='viridis')
axes[2].set_title('Average Pooling 2x2')
axes[2].set_xticks(range(2))
axes[2].set_yticks(range(2))

plt.tight_layout()
plt.savefig('/tmp/pooling_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print('Max Pooling:')
print(max_out)
print()
print('Average Pooling:')
print(avg_out)


### O que observar

1. Max pooling preserva os **valores mais altos** (features mais ativadas)
2. Average pooling suaviza, sendo util para classificacao global
3. Uma janela 2x2 com stride 2 reduz cada dimensao pela metade (4x4 -> 2x2)
4. Max pooling: [6, 8, 14, 16] -- pegou os maximos de cada quadrante
5. Avg pooling: [3.5, 5.5, 11.5, 13.5] -- media de cada quadrante

### O que concluir

Max pooling e o padrao mais usado porque preserva ativacoes fortes (features detectadas).
Average pooling e mais usado em camadas finais (Global Average Pooling substitui
camadas fully connected em arquiteturas modernas). A escolha entre eles depende
da tarefa: max para deteccao de features, average para representacoes globais.

### Conexao com outros notebooks

O conceito de reduzir dimensionalidade mantendo informacao essencial apareceu em
3_6 (reducao de dimensionalidade com PCA). Pooling faz algo analogo no espaco espacial:
comprime a imagem mantendo as features mais relevantes.

## 4. Arquiteturas Classicas: A Evolucao das CNNs

### Analogia: Evolucao de Predios

Pense nas arquiteturas CNN como evolucao de predios:
- **LeNet (1998):** casa de 2 andares -- simples mas funcional
- **AlexNet (2012):** predio de 8 andares -- mais profundo, GPU pela primeira vez
- **VGG (2014):** arranha-ceu de 19 andares -- muito profundo, blocos repetitivos
- **GoogLeNet (2014):** predio com andares de larguras diferentes -- multiplas escalas

### Por que em ML: Cada Salto Arquitetural Desbloqueou Novas Capacidades

LeNet reconhecia digitos. AlexNet reconhecia 1000 categorias. VGG mostrou que
profundidade importa. GoogLeNet mostrou que *largura* tambem importa.
A evolucao dessas arquiteturas e a historia da visao computacional moderna.

In [ ]:
# Comparacao de arquiteturas: parametros e complexidade
architectures = {
    'LeNet-5 (1998)': {
        'layers': 7, 'params': 60_000, 'top5_imagenet': None,
        'innovation': 'Primeira CNN bem-sucedida'
    },
    'AlexNet (2012)': {
        'layers': 8, 'params': 60_000_000, 'top5_imagenet': 16.4,
        'innovation': 'ReLU + GPU + Dropout'
    },
    'VGG-16 (2014)': {
        'layers': 16, 'params': 138_000_000, 'top5_imagenet': 7.3,
        'innovation': 'Kernels 3x3 empilhados'
    },
    'GoogLeNet (2014)': {
        'layers': 22, 'params': 6_800_000, 'top5_imagenet': 6.7,
        'innovation': 'Modulos Inception (multi-escala)'
    },
    'ResNet-50 (2015)': {
        'layers': 50, 'params': 25_600_000, 'top5_imagenet': 3.6,
        'innovation': 'Skip connections'
    },
    'EfficientNet-B0 (2019)': {
        'layers': 18, 'params': 5_300_000, 'top5_imagenet': 2.9,
        'innovation': 'Compound scaling'
    },
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(architectures.keys())
params = [a['params'] for a in architectures.values()]
layers = [a['layers'] for a in architectures.values()]
top5 = [a['top5_imagenet'] for a in architectures.values()]

# Grafico 1: Parametros
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
bars = axes[0].barh(names, [p/1e6 for p in params], color=colors)
axes[0].set_xlabel('Parametros (milhoes)')
axes[0].set_title('Numero de Parametros por Arquitetura')
for bar, p in zip(bars, params):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{p/1e6:.1f}M', va='center', fontsize=9)

# Grafico 2: Erro ImageNet (excluindo LeNet que nao tem)
valid = [(n, t) for n, t in zip(names[1:], top5[1:]) if t is not None]
ax_names, ax_top5 = zip(*valid)
axes[1].plot(ax_names, ax_top5, 'ro-', markersize=10, linewidth=2)
axes[1].set_ylabel('Top-5 Error (%)')
axes[1].set_title('Evolucao do Erro no ImageNet')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/cnn_architectures.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nInovacoes por arquitetura:')
for name, info in architectures.items():
    print(f'  {name}: {info["innovation"]}')

### O que observar

1. VGG-16 tem **138M parametros** -- e o maior modelo, mostrando que mais parametros nao = melhor
2. GoogLeNet tem apenas **6.8M parametros** mas erro menor que VGG -- design inteligente importa mais
3. EfficientNet-B0 tem **5.3M parametros** e o menor erro -- eficiencia moderna
4. O erro no ImageNet caiu de 16.4% (2012) para 2.9% (2019) -- progresso exponencial
5. ResNet com 50 camadas funciona gracas a skip connections (veremos na proxima secao)

### O que concluir

A historia das CNNs mostra que **profundidade sozinha nao basta** -- design inteligente
(Inception, skip connections, compound scaling) permite modelos menores e mais precisos.
A tendencia moderna e buscar o melhor trade-off entre parametros e performance.

### Conexao com outros notebooks

Em 4_2 (arquiteturas deep), vimos conceitos de camadas convolucionais e recorrentes.
Aqui, vemos como esses conceitos se materializam em arquiteturas reais. A comparacao
de parametros conecta com 4_6 (otimizacao), onde vimos que eficiencia computacional importa.

## 5. ResNet e Skip Connections

### Analogia: Escada de Emergencia

Imagine um predio de 50 andares. Se o elevador (gradient flow) quebrar num andar
intermediario, ninguem acima consegue descer. A **skip connection** e como uma escada
de emergencia: mesmo se um andar "bloquear" o gradiente, ha um caminho alternativo.

### Definicao Formal

Um **residual block** computa: y = F(x) + x

Em vez de aprender a funcao completa F(x), a rede aprende o **residual** F(x) = y - x.
Se o bloco nao contribui, F(x) converge para zero e o output e simplesmente x (identidade).

### Por que em ML: O Problema da Degradacao

Redes muito profundas (>20 camadas) sofrem com **degradacao**: o treinamento fica *pior*
(nao e overfitting -- o erro de treino *tambem* sobe). Skip connections resolvem isso
garantindo que o gradiente flua mesmo em redes com 152+ camadas.

In [ ]:
# Demonstracao visual: Residual Block vs Plain Block
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plain Block (sem skip)
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Plain Block (sem skip connection)', fontsize=13, fontweight='bold')

blocks_plain = [
    (3, 8, 4, 1.2, 'Input x', '#3498db'),
    (3, 5.5, 4, 1.2, 'Conv + BN + ReLU', '#e74c3c'),
    (3, 3, 4, 1.2, 'Conv + BN', '#e74c3c'),
    (3, 0.5, 4, 1.2, 'ReLU -> y = F(x)', '#2ecc71'),
]
for x, y, w, h, label, color in blocks_plain:
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                 facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Setas
for y_start, y_end in [(8, 6.7), (5.5, 4.2), (3, 1.7)]:
    ax.annotate('', xy=(5, y_end), xytext=(5, y_start),
                arrowprops=dict(arrowstyle='->', lw=2))
ax.axis('off')

# Residual Block (com skip)
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Residual Block (com skip connection)', fontsize=13, fontweight='bold')

blocks_res = [
    (3, 8, 4, 1.2, 'Input x', '#3498db'),
    (3, 5.5, 4, 1.2, 'Conv + BN + ReLU', '#e74c3c'),
    (3, 3, 4, 1.2, 'Conv + BN', '#e74c3c'),
    (3, 0.5, 4, 1.2, 'ReLU -> y = F(x) + x', '#2ecc71'),
]
for x, y, w, h, label, color in blocks_res:
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                 facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Setas normais
for y_start, y_end in [(8, 6.7), (5.5, 4.2), (3, 1.7)]:
    ax.annotate('', xy=(5, y_end), xytext=(5, y_start),
                arrowprops=dict(arrowstyle='->', lw=2))

# Skip connection (curva)
ax.annotate('', xy=(7.5, 1.1), xytext=(7.5, 8.6),
            arrowprops=dict(arrowstyle='->', lw=3, color='#f39c12',
                           connectionstyle='arc3,rad=0.3'))
ax.text(9, 5, 'Skip\n(identidade)', ha='center', va='center',
        fontsize=11, color='#f39c12', fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/residual_block.png', dpi=100, bbox_inches='tight')
plt.show()

print('Esquerda: bloco sem skip -- gradiente pode desaparecer em redes profundas')
print('Direita: bloco residual -- gradiente flui pela skip connection')
print()
print('Intuicao: se F(x) nao ajuda, o output = x (identidade)')
print('Isso garante que camadas profundas NUNCA degradem a performance')

### O que observar

1. No plain block, o gradiente precisa atravessar TODAS as camadas -- pode desaparecer
2. No residual block, o gradiente tem um "atalho" direto pela skip connection
3. A soma F(x) + x e simples e nao adiciona parametros extras
4. Se F(x) = 0 (a rede aprende a "desligar" o bloco), y = x -- nenhum dano
5. Isso explica por que ResNet-152 funciona melhor que VGG-19 mesmo sendo 8x mais profunda

### O que concluir

Skip connections sao uma das inovacoes mais importantes da historia de deep learning.
Elas resolvem o problema da degradacao e permitem treinar redes arbitrariamente profundas.
O conceito foi tao influente que aparece em praticamente todas as arquiteturas modernas
(DenseNet, U-Net, Transformers).

### Conexao com outros notebooks

Em 4_3 (treinamento deep), estudamos gradient vanishing e tecnicas como BatchNorm.
Skip connections sao outra solucao para o mesmo problema -- e complementar ao BatchNorm.
Na secao seguinte, veremos como arquiteturas modernas combinam ambas as tecnicas.

## 6. Arquiteturas Modernas e Tendencias

### Analogia: Evolucao dos Carros

- **DenseNet:** carro com multiplos espelhos retrovisores -- cada camada ve *todas* as anteriores
- **EfficientNet:** carro com motor turbo eficiente -- escala profundidade, largura e resolucao juntos
- **ConvNeXt:** carro classico com motor moderno -- ResNet redesenhado com truques de Transformers

### Por que em ML: O Trade-off Accuracy vs Eficiencia

Na pratica, nao basta ter o modelo mais preciso -- ele precisa rodar no hardware disponivel.
EfficientNet e ConvNeXt mostram que design inteligente permite modelos menores e mais rapidos
sem sacrificar accuracy. Isso e essencial para deploy em celulares e edge devices.

In [ ]:
# Visualizacao: DenseNet vs ResNet connectivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ResNet connectivity (sequential with skip)
ax = axes[0]
ax.set_title('ResNet: Skip Connections', fontsize=12, fontweight='bold')
n_layers = 5
positions = [(2, i*1.5) for i in range(n_layers)]

for i, (x, y) in enumerate(positions):
    ax.add_patch(patches.Circle((x, y), 0.4, facecolor='#3498db', edgecolor='black'))
    ax.text(x, y, f'L{i+1}', ha='center', va='center', fontweight='bold', color='white')

# Conexoes sequenciais
for i in range(n_layers - 1):
    ax.annotate('', xy=(2, positions[i+1][1] - 0.4), xytext=(2, positions[i][1] + 0.4),
                arrowprops=dict(arrowstyle='->', lw=1.5))

# Skip connections
for i in range(0, n_layers - 2, 2):
    ax.annotate('', xy=(2.5, positions[i+2][1]), xytext=(2.5, positions[i][1]),
                arrowprops=dict(arrowstyle='->', lw=2, color='#e74c3c',
                               connectionstyle='arc3,rad=0.5'))

ax.set_xlim(0, 4)
ax.set_ylim(-1, n_layers * 1.5)
ax.axis('off')

# DenseNet connectivity (all-to-all)
ax = axes[1]
ax.set_title('DenseNet: Dense Connections', fontsize=12, fontweight='bold')

for i, (x, y) in enumerate(positions):
    ax.add_patch(patches.Circle((x, y), 0.4, facecolor='#2ecc71', edgecolor='black'))
    ax.text(x, y, f'L{i+1}', ha='center', va='center', fontweight='bold', color='white')

# Todas as conexoes
colors_dense = plt.cm.Reds(np.linspace(0.3, 0.9, n_layers))
for i in range(n_layers):
    for j in range(i + 1, n_layers):
        offset = 0.3 + (j - i) * 0.15
        ax.annotate('', xy=(2 + offset, positions[j][1]),
                    xytext=(2 + offset, positions[i][1]),
                    arrowprops=dict(arrowstyle='->', lw=1, color=colors_dense[j-i],
                                   connectionstyle=f'arc3,rad={0.3 + (j-i)*0.1}',
                                   alpha=0.7))

ax.set_xlim(0, 5)
ax.set_ylim(-1, n_layers * 1.5)
ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/dense_vs_resnet.png', dpi=100, bbox_inches='tight')
plt.show()

# EfficientNet compound scaling
print('EfficientNet Compound Scaling:')
print('  Resolucao x Profundidade x Largura = Performance')
print()
for variant, res, depth, width, params, top1 in [
    ('B0', 224, 1.0, 1.0, 5.3, 77.1),
    ('B1', 240, 1.1, 1.0, 7.8, 79.1),
    ('B3', 300, 1.4, 1.2, 12.0, 81.6),
    ('B5', 456, 1.8, 1.6, 30.0, 83.6),
    ('B7', 600, 2.2, 2.0, 66.0, 84.3),
]:
    print(f'  {variant}: {res}px, depth={depth}x, width={width}x '
          f'-> {params}M params, Top-1={top1}%')

### O que observar

1. ResNet conecta camadas com **saltos** de 2 em 2 -- cada bloco tem um atalho
2. DenseNet conecta **todas** as camadas entre si -- cada camada recebe input de todas anteriores
3. DenseNet promove **reuso de features** -- menos parametros para mesma capacidade
4. EfficientNet escala **tres dimensoes juntas** (profundidade, largura, resolucao)
5. Escalar apenas uma dimensao (so profundidade OU so largura) e sub-otimo

### O que concluir

Arquiteturas modernas nao sao apenas "mais profundas" -- sao **mais inteligentes**.
DenseNet maximiza reuso. EfficientNet encontra o balance otimo entre dimensoes.
ConvNeXt mostra que a arquitetura ResNet ainda pode competir com Transformers
quando modernizada. A escolha de arquitetura depende do constraint: memoria, latencia,
ou accuracy.

### Conexao com outros notebooks

A ideia de compound scaling do EfficientNet conecta com 4_5 (aceleracao hardware),
onde vimos que constraints de hardware influenciam decisoes de modelo. Em 5A_5
(Vision Transformers), veremos como Transformers desafiam CNNs e como ConvNeXt responde.

## 7. Data Augmentation para Visao

### Analogia: Variacao de Fotos de Passaporte

Para treinar um sistema de reconhecimento facial, voce nao tira apenas uma foto --
tira fotos com diferentes angulos, iluminacoes, e expressoes. Data augmentation
faz isso **sinteticamente**: cria variacoes da mesma imagem para ensinar a rede
a ser **invariante** a essas transformacoes.

### Por que em ML: Mais Dados Sem Mais Coleta

Coletar e anotar imagens e caro. Augmentation multiplica o dataset virtualmente,
reduzindo overfitting e melhorando generalizacao. Tecnicas avancadas como Mixup
e CutMix vao alem de transformacoes geometricas, criando novas amostras sinteticas.

In [ ]:
# Demonstracao de augmentations com NumPy puro
np.random.seed(42)

# Criar imagem sintetica simples (digito "7")
img = np.zeros((28, 28))
img[5:8, 8:22] = 1.0       # barra horizontal
img[7:25, 18:21] = 1.0     # barra vertical

def rotate_image(img, angle_deg):
    """Rotacao simples por interpolacao nearest"""
    h, w = img.shape
    center = np.array([h/2, w/2])
    angle = np.radians(angle_deg)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    rotated = np.zeros_like(img)
    for i in range(h):
        for j in range(w):
            y, x = i - center[0], j - center[1]
            src_y = int(cos_a * y + sin_a * x + center[0])
            src_x = int(-sin_a * y + cos_a * x + center[1])
            if 0 <= src_y < h and 0 <= src_x < w:
                rotated[i, j] = img[src_y, src_x]
    return rotated

def translate_image(img, dy, dx):
    """Translacao simples"""
    h, w = img.shape
    result = np.zeros_like(img)
    for i in range(h):
        for j in range(w):
            si, sj = i - dy, j - dx
            if 0 <= si < h and 0 <= sj < w:
                result[i, j] = img[si, sj]
    return result

def add_noise(img, std=0.1):
    return np.clip(img + np.random.randn(*img.shape) * std, 0, 1)

def flip_horizontal(img):
    return img[:, ::-1]

# Aplicar augmentations
augmentations = {
    'Original': img,
    'Rotacao 15': rotate_image(img, 15),
    'Rotacao -10': rotate_image(img, -10),
    'Translacao': translate_image(img, 3, -2),
    'Ruido': add_noise(img, 0.15),
    'Flip H': flip_horizontal(img),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, aug_img) in zip(axes.flat, augmentations.items()):
    ax.imshow(aug_img, cmap='gray')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Data Augmentation -- Variacoes de uma Mesma Imagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/augmentation_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print('Cada variacao ensina a rede que o "7" continua sendo "7" apesar de:')
print('  - Pequenas rotacoes (15 graus)')
print('  - Deslocamentos (translacao)')
print('  - Ruido (camera de baixa qualidade)')
print('  - Espelhamento horizontal')

### O que observar

1. Todas as variacoes mantem a **identidade** do digito "7"
2. Rotacoes pequenas (10-15 graus) sao realistas -- simular cameras nao-alinhadas
3. Ruido simula cameras de baixa qualidade ou condicoes adversas
4. Flip horizontal pode nao fazer sentido para todos os digitos (6 vs 9!)
5. A escolha de augmentations depende do **dominio**: raio-X medico nao deve ser flipado

### O que concluir

Data augmentation e uma tecnica de **regularizacao** que aumenta virtualmente o dataset.
Deve ser aplicada **apenas no treino** (nunca na validacao/teste). A escolha de quais
augmentations usar depende do que faz sentido no dominio -- augmentations irrealistas
podem prejudicar o modelo.

### Conexao com outros notebooks

Em 4_3 (treinamento deep), vimos augmentation como tecnica de regularizacao junto com
Dropout e BatchNorm. Aqui, aprofundamos as tecnicas especificas para imagens. Em 5A_2
(classificacao de imagens), aplicaremos essas tecnicas em pipelines completos de treino.

### O que observar sobre o Design de Arquiteturas CNN

Ao projetar ou escolher uma arquitetura CNN, considere:
1. **Complexidade do problema:** digitos (LeNet) vs 1000 classes (ResNet/EfficientNet)
2. **Budget computacional:** mobile (MobileNet) vs datacenter (ResNet-152)
3. **Tamanho do dataset:** pouco dado -> transfer learning e augmentation pesada
4. **Latencia requerida:** real-time (EfficientNet-B0) vs batch processing (EfficientNet-B7)

### O que concluir sobre a Evolucao de CNNs

A evolucao de LeNet a EfficientNet mostra um padrao claro: **modelos mais inteligentes,
nao apenas maiores**. Cada inovacao (ReLU, BatchNorm, skip connections, compound scaling)
resolveu um problema especifico. Entender essas motivacoes e mais importante que memorizar
arquiteturas.

### Conexao com outros notebooks sobre CNN em Producao

Na pratica profissional, raramente treinamos CNNs do zero. Transfer learning (4_4) com
arquiteturas pre-treinadas no ImageNet e o padrao. Em 5A_5 (Vision Transformers), veremos
como a atencao global pode superar a convolucao local em certos cenarios.

## 8. Exercicios Praticos

### Exercicio 1: Convoluir com Diferentes Kernels

Aplique a funcao `conv2d_manual` com tres kernels diferentes numa imagem 6x6
e compare os resultados. Identifique qual kernel detecta bordas horizontais,
qual detecta bordas verticais e qual funciona como blur.

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Aplicar diferentes kernels
# Crie uma imagem 6x6 com um padrao simples
img_6x6 = np.array([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
], dtype=np.float32)

# Kernels para testar
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
blur = np.ones((3, 3), dtype=np.float32) / 9.0

# TAREFA DO ALUNO: Aplique conv2d_manual para cada kernel e visualize
# resultado_x = conv2d_manual(img_6x6, ???)
# resultado_y = conv2d_manual(img_6x6, ???)
# resultado_blur = conv2d_manual(img_6x6, ???)
resultado_x = None
resultado_y = None
resultado_blur = None

In [ ]:
# SOLUCAO - Exercicio 1
# Definir variaveis (caso a celula de pratica nao tenha sido executada)
img_6x6 = np.array([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
], dtype=np.float32)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
blur = np.ones((3, 3), dtype=np.float32) / 9.0

resultado_x = conv2d_manual(img_6x6, sobel_x)
resultado_y = conv2d_manual(img_6x6, sobel_y)
resultado_blur = conv2d_manual(img_6x6, blur)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, data) in zip(axes, [
    ('Original', img_6x6),
    ('Sobel X (bordas V)', resultado_x),
    ('Sobel Y (bordas H)', resultado_y),
    ('Blur (media)', resultado_blur),
]):
    im = ax.imshow(data, cmap='RdBu_r', vmin=-4, vmax=4)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.axis('off')
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i,j]:.1f}', ha='center', va='center', fontsize=8)

plt.colorbar(im, ax=axes, shrink=0.8)
plt.tight_layout()
plt.savefig('/tmp/kernels_exercise.png', dpi=100, bbox_inches='tight')
plt.show()

print('Sobel X detecta bordas VERTICAIS (transicoes esquerda-direita)')
print('Sobel Y detecta bordas HORIZONTAIS (transicoes cima-baixo)')
print('Blur suaviza a imagem (media local)')

### Exercicio 2: Calcular Dimensoes de Saida

Dada uma imagem 32x32 RGB (3 canais) passando por uma sequencia de camadas CNN,
calcule manualmente as dimensoes em cada etapa.

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Calcular dimensoes
# Sequencia de camadas:
# 1. Conv2d(3, 16, kernel=3, stride=1, padding=1)
# 2. MaxPool2d(2, 2)
# 3. Conv2d(16, 32, kernel=3, stride=1, padding=1)
# 4. MaxPool2d(2, 2)
# 5. Conv2d(32, 64, kernel=3, stride=2, padding=0)

# Preencha as dimensoes em cada etapa
# Use a formula: H_out = (H_in + 2*p - k) / s + 1
input_shape = (3, 32, 32)  # (C, H, W)

# TAREFA DO ALUNO: Calcule
# apos_conv1 = (None, None, None)
# apos_pool1 = (None, None, None)
# apos_conv2 = (None, None, None)
# apos_pool2 = (None, None, None)
# apos_conv3 = (None, None, None)
apos_conv1 = None
apos_pool1 = None
apos_conv2 = None
apos_pool2 = None
apos_conv3 = None

In [ ]:
# SOLUCAO - Exercicio 2
def calc_dim(h_in, k, s, p):
    return (h_in + 2*p - k) // s + 1

print('Calculo de dimensoes passo a passo:')
print(f'{"Camada":<35} {"Output Shape":<20} {"Calculo"}')
print('-' * 80)

# Input
h = 32
print(f'{"Input":<35} {"(3, 32, 32)":<20}')

# Conv1: (3, 32, 32) -> (16, 32, 32)
h = calc_dim(32, 3, 1, 1)
apos_conv1 = (16, h, h)
print(f'{"Conv2d(3,16, k=3, s=1, p=1)":<35} {str(apos_conv1):<20} (32+2-3)/1+1 = 32')

# Pool1: (16, 32, 32) -> (16, 16, 16)
h = calc_dim(h, 2, 2, 0)
apos_pool1 = (16, h, h)
print(f'{"MaxPool2d(2, 2)":<35} {str(apos_pool1):<20} (32-2)/2+1 = 16')

# Conv2: (16, 16, 16) -> (32, 16, 16)
h = calc_dim(h, 3, 1, 1)
apos_conv2 = (32, h, h)
print(f'{"Conv2d(16,32, k=3, s=1, p=1)":<35} {str(apos_conv2):<20} (16+2-3)/1+1 = 16')

# Pool2: (32, 16, 16) -> (32, 8, 8)
h = calc_dim(h, 2, 2, 0)
apos_pool2 = (32, h, h)
print(f'{"MaxPool2d(2, 2)":<35} {str(apos_pool2):<20} (16-2)/2+1 = 8')

# Conv3: (32, 8, 8) -> (64, 3, 3)
h = calc_dim(h, 3, 2, 0)
apos_conv3 = (64, h, h)
print(f'{"Conv2d(32,64, k=3, s=2, p=0)":<35} {str(apos_conv3):<20} (8+0-3)/2+1 = 3')

print()
print(f'Parametros para flatten: 64 * 3 * 3 = {64*3*3} neuronios')

### Exercicio 3: Comparar Pooling Strategies

Aplique max pooling e average pooling na mesma imagem e analise qual preserva
melhor as features de borda. Depois implemente global average pooling.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Comparar pooling
# Crie uma imagem 8x8 com bordas claras
img_8x8 = np.zeros((8, 8))
img_8x8[2:6, 2:6] = 1.0  # quadrado central

# TAREFA DO ALUNO: Aplique max_pool2d e avg_pool2d com kernel=2, stride=2
# max_result = max_pool2d(???)
# avg_result = avg_pool2d(???)
# global_avg = ???  # Media de toda a imagem
max_result = None
avg_result = None
global_avg = None

In [ ]:
# SOLUCAO - Exercicio 3
# Definir variaveis (caso a celula de pratica nao tenha sido executada)
img_8x8 = np.zeros((8, 8))
img_8x8[2:6, 2:6] = 1.0

max_result = max_pool2d(img_8x8, kernel_size=2, stride=2)
avg_result = avg_pool2d(img_8x8, kernel_size=2, stride=2)
global_avg = np.mean(img_8x8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (name, data) in zip(axes, [
    ('Original 8x8', img_8x8),
    ('Max Pool 4x4', max_result),
    ('Avg Pool 4x4', avg_result),
]):
    ax.imshow(data, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(name, fontsize=12, fontweight='bold')
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/pooling_exercise.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Global Average Pooling: {global_avg:.4f}')
print()
print('Max pooling preserva melhor as bordas (1.0 onde ha features)')
print('Avg pooling suaviza (0.25 e 0.5 nas bordas)')
print('Global avg reduz tudo a um escalar -- usado em classificacao final')

### O que observar sobre Implementacao Pratica de CNNs

1. Frameworks modernos (PyTorch, TensorFlow) abstraem a convolucao, mas entender
   a operacao manual ajuda a debugar problemas de dimensao
2. A escolha entre `padding='same'` e `padding='valid'` afeta dimensoes de saida
3. Depthwise Separable Convolutions (MobileNet) reduzem computacao em ate 9x

### O que concluir sobre Trade-offs em Producao

Na pratica, o modelo "ideal" depende do cenario de deploy. Um modelo com 99% accuracy
que leva 2 segundos por imagem pode ser pior que um com 95% que responde em 50ms.
EfficientNet-B0 e MobileNet sao escolhas populares para mobile; ResNet-50 para servidores.

### Conexao com outros notebooks sobre Eficiencia

A escolha de arquitetura conecta diretamente com 4_5 (aceleracao hardware) -- modelos
menores usam menos memoria GPU e permitem batch sizes maiores. Tambem conecta com
6_1 (deploy), onde o tamanho do modelo afeta latencia e custo de servico.

### Por que em ML: CNNs como Extractores de Features Universais

Features aprendidas por CNNs em ImageNet sao tao boas que funcionam como "features universais"
para muitas tarefas: classificacao, deteccao, segmentacao, e ate tarefas nao-visuais
(espectrogramas de audio, representacoes de texto como imagens).

### O que observar sobre a Transicao CNNs -> Transformers

Vision Transformers (ViT) mostraram que atencao global pode superar convolucao local,
especialmente em datasets grandes. Porem, CNNs continuam superiores em datasets menores
e em cenarios com restricoes de compute. Hibridos (CNN + Transformer) sao o estado da arte.

### O que concluir sobre o Futuro da Visao Computacional

CNNs nao estao "mortas" -- estao evoluindo. ConvNeXt (2022) mostrou que uma ResNet
modernizada compete com Transformers. A tendencia e modelos hibridos que combinam
o melhor dos dois mundos: localidade (CNN) + contexto global (Transformer).

### Conexao com outros notebooks sobre Vision Transformers

Em 5A_5 (Vision Transformers), estudaremos como ViT e Swin Transformer funcionam.
A base que construimos aqui (convolucao, pooling, features hierarquicas) e essencial
para entender *por que* Transformers precisam de adaptacoes especiais para imagens.

### Por que em ML: Transfer Learning como Paradigma Dominante

Na pratica, quase ninguem treina CNNs do zero. O paradigma dominante e:
1. Pegar modelo pre-treinado no ImageNet (ResNet, EfficientNet)
2. Congelar camadas iniciais (features genericas)
3. Fine-tunar camadas finais para a tarefa especifica
Isso funciona porque as primeiras camadas aprendem features universais (bordas, texturas).

## 9. Erros Comuns e Armadilhas

### Erro 1: Dimensoes Incompativeis entre Camadas
**Sintoma:** RuntimeError: size mismatch ao conectar conv com linear
**Causa:** Nao calcular corretamente H_out * W_out * C apos as camadas conv
**Solucao:** Usar `print(x.shape)` apos cada camada durante debug, ou
`nn.AdaptiveAvgPool2d((1,1))` antes do classificador

### Erro 2: Esquecer model.eval() durante Validacao
**Sintoma:** Performance diferente entre treino e teste com BatchNorm/Dropout
**Causa:** BatchNorm usa estatisticas do batch em modo treino vs populacao em eval
**Solucao:** Sempre `model.train()` para treino e `model.eval()` para inferencia

### Erro 3: Data Augmentation no Conjunto de Teste
**Sintoma:** Metricas de teste instáveis entre execucoes
**Causa:** Aplicar transformacoes aleatorias (rotacao, flip) nos dados de teste
**Solucao:** Augmentation APENAS no treino; teste usa apenas resize + normalize

### Erro 4: Padding Incorreto Muda Dimensoes Inesperadamente
**Sintoma:** Feature maps com tamanho diferente do esperado
**Causa:** padding=0 quando deveria ser padding=1 (ou vice-versa)
**Solucao:** Para manter dimensao com kernel 3x3: padding=1; com 5x5: padding=2

### Erro 5: Learning Rate Muito Alto Destroi Pesos Pre-treinados
**Sintoma:** Loss sobe ou oscila muito no inicio do fine-tuning
**Causa:** LR alta demais para pesos ja otimizados
**Solucao:** Comece com LR 10-100x menor que treino do zero (ex: 1e-4 vs 1e-2)

### Erro 6: Confundir Channels-First vs Channels-Last
**Sintoma:** Erro de dimensao ao alimentar imagem na rede
**Causa:** PyTorch usa (B, C, H, W) mas imagens PIL/OpenCV sao (H, W, C)
**Solucao:** Use `transforms.ToTensor()` que converte automaticamente

## 10. Resumo e Conexoes

### Hierarquia de Conceitos

```
CNNs Fundamentos
|
|-- Convolucao
|   |-- Kernel/Filtro (detector de padroes)
|   |-- Stride (passo do filtro)
|   |-- Padding (bordas)
|   |-- Receptive Field (campo de visao)
|
|-- Pooling
|   |-- Max Pooling (preserva features fortes)
|   |-- Average Pooling (representacao global)
|   |-- Global Pooling (substitui FC layers)
|
|-- Arquiteturas
|   |-- LeNet (1998) -- pioneira
|   |-- AlexNet (2012) -- GPU + ReLU + Dropout
|   |-- VGG (2014) -- profundidade com 3x3
|   |-- ResNet (2015) -- skip connections
|   |-- EfficientNet (2019) -- compound scaling
|
|-- Tecnicas
    |-- Data Augmentation (regularizacao visual)
    |-- Transfer Learning (reusar features)
    |-- Feature Visualization (interpretabilidade)
```

### Tabela de Conexoes

| Conceito | Conexao | Notebook |
|----------|---------|----------|
| Compartilhamento de pesos | Regularizacao implicita | 4_3 treinamento_deep |
| Skip connections | Gradient flow | 4_3 treinamento_deep |
| Reducao dimensionalidade (pooling) | PCA e projecoes | 3_6 reducao_dimensionalidade |
| Data augmentation | Regularizacao | 4_3 treinamento_deep |
| Parameter counting | Complexidade computacional | 4_6 otimizacao_python |
| Transfer learning | Reusar features pre-treinadas | 4_4 transfer_learning |
| Compound scaling | Hardware constraints | 4_5 aceleracao_hardware |
| Convolucao como produto interno local | Algebra linear | 0_3 algebra_linear_matrizes |

### Checklist de Competencias

- [ ] Sei explicar por que CNNs sao superiores a MLPs para imagens
- [ ] Consigo calcular dimensoes de saida dado kernel, stride e padding
- [ ] Implemento convolucao e pooling manualmente com NumPy
- [ ] Conheco a evolucao das arquiteturas (LeNet -> EfficientNet) e suas motivacoes
- [ ] Entendo por que skip connections permitem redes profundas
- [ ] Sei escolher augmentations apropriadas para um dominio
- [ ] Consigo comparar arquiteturas em termos de parametros vs accuracy

### Proximos Passos

No proximo notebook (5A_2 -- Classificacao de Imagens), aplicaremos esses fundamentos
num pipeline completo de classificacao: dataset real, treino com augmentation, avaliacao
com metricas adequadas, e analise de erros com confusion matrix.